# Where does the time of one halo go? (TNG50-2)

The core scan (`03_cores_scaling_TNG50-2.ipynb`) saturates at ~120 halos/h from N ≈ 16 workers.
Here we profile **one halo** with `cProfile`, running the pipeline's own worker
(`process_single_halo`, `HCD_only`, 363 skewers, 1 OpenMP thread) and split its time into:
* **reading** the snapshot (HDF5) and selecting the particles near the skewers;
* **interpolation** of the particles onto the skewers (C code: tau, colden, temp);
* **HCD decomposition** (fork, Python, per skewer);
* the rest (post-processing in `tools/`).

If reading dominates, extracting many halos in one `Spectra` object would help;
if not, the gain is small.

Halo: the warm-up halo of notebook 03 (log M200 in [11, 11.5), ~3.4 min alone).

In [1]:
import os
# One OpenMP thread, as in the omp1 points of notebook 03.
# Must be set before numpy / fake_spectra are imported.
os.environ["OMP_NUM_THREADS"] = "1"

import sys
import time
import cProfile
import pstats
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib widget
%load_ext autoreload
%autoreload 2

In [2]:
# ==============================================================================
# CELL 1: CONFIGURATION
# ==============================================================================
# Project root = the folder that contains tools/
PROJ_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "tools").is_dir())
BENCH_DIR = PROJ_ROOT / "benchmarks"

SIM_NAME = "TNG50-2"
SNAP_NUM = 25

# Same halo as the warm-up of notebook 03
HALO_ID = int(np.load(BENCH_DIR / "results" / f"{SIM_NAME}_cores_scaling" / "warmup.npy")[0])

RERUN = False      # False: reuse the .prof file already in RESULTS_DIR

RESULTS_DIR = BENCH_DIR / "results" / f"{SIM_NAME}_profile"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
PROF_FILE = RESULTS_DIR / f"halo{HALO_ID}.prof"

# The pipeline module writes its error log in the cwd at import: keep it here
os.chdir(RESULTS_DIR)
sys.path.insert(0, str(PROJ_ROOT / "tools"))
import run_parallel_illustris as rpi

# Same worker configuration as the production runs and notebook 03
cfg = {
    "base_path": f"/home/STORAGE/{SIM_NAME}",
    "snap_num": SNAP_NUM,
    "sim_type": "TNG",
    "modes": ("HCD_only",),
    "r_multipliers": rpi.build_r_multipliers(30, 0.01, 50.0),
}
print("halo    :", HALO_ID)
print("results :", RESULTS_DIR)

halo    : 852
results : /home/vroberto/FakeSpectra_DLAs/benchmarks/results/TNG50-2_profile


## 1. Profile one halo
About 4 min (cProfile adds some overhead to the Python parts).

In [3]:
# ==============================================================================
# CELL 2: RUN THE WORKER UNDER cProfile
# ==============================================================================
if RERUN or not PROF_FILE.exists():
    pr = cProfile.Profile()
    t0 = time.time()
    pr.enable()
    _, rows, _, status = rpi.process_single_halo((HALO_ID, 42, cfg))
    pr.disable()
    print(f"{status}: {len(rows)} skewers in {(time.time() - t0) / 60:.1f} min")
    pr.dump_stats(PROF_FILE)

stats = pstats.Stats(str(PROF_FILE))
print(f"total profiled time: {stats.total_tt / 60:.1f} min")

--> [Halo 852] extracting 363 skewers (HCD_only) in RAM...
SUCCESS: 363 skewers in 4.8 min
total profiled time: 4.8 min


## 2. Breakdown
Cumulative time of the main functions. The blocks overlap (e.g. reading is inside each
product), so they do not add up to the total.

In [4]:
# ==============================================================================
# CELL 3: TIME OF THE MAIN FUNCTIONS
# ==============================================================================
def cum_time(file_end, func):
    # Cumulative time [s] and number of calls of `func` defined in a file ending with `file_end`
    ct, nc = 0.0, 0
    for (fname, _, name), (_, ncalls, _, cumt, _) in stats.stats.items():
        if fname.endswith(file_end) and name == func:
            ct += cumt
            nc += ncalls
    return ct, nc

# (label, file, function): what each block of the halo does
BLOCKS = [
    ("whole halo",                       "run_parallel_illustris.py", "process_single_halo"),
    ("  tau pass",                       "/spectra.py",               "get_tau"),
    ("  colden pass",                    "/spectra.py",               "get_col_density"),
    ("  temp pass",                      "/spectra.py",               "get_temp"),
    ("reading + particle selection",     "/spectra.py",               "_read_particle_data"),
    ("  HDF5 reads",                     "abstractsnapshot.py",       "get_data"),
    ("  particles near the skewers",     "/spectra.py",               "particles_near_lines"),
    ("interpolation (C)",                "/spectra.py",               "_do_interpolation_work"),
    ("HCD decomposition",                "/spectra.py",               "_hcd_peak_mask"),
    ("post-processing (centring, cut)",  "utils.py",                  "get_centered_spectrum"),
]

total = cum_time("run_parallel_illustris.py", "process_single_halo")[0]
rows_t = []
for label, file_end, func in BLOCKS:
    ct, nc = cum_time(file_end, func)
    rows_t.append(dict(block=label, function=func, calls=nc,
                       time_min=ct / 60, frac_halo=ct / total))
breakdown = pd.DataFrame(rows_t)
breakdown.round(3)

,block,function,calls,time_min,frac_halo
0,whole halo,process_single_halo,1,4.841,1.000
1,tau pass,get_tau,1,4.838,0.999
2,colden pass,get_col_density,2,0.000,0.000
3,temp pass,get_temp,1,0.000,0.000
4,reading + particle selection,_read_particle_data,128,4.605,0.951
5,HDF5 reads,get_data,1390,1.998,0.413
6,particles near the skewers,particles_near_lines,491,0.539,0.111
7,interpolation (C),_do_interpolation_work,866,0.201,0.042
8,HCD decomposition,_hcd_peak_mask,363,0.110,0.023
9,"post-processing (centring, cut)",get_centered_spectrum,363,0.000,0.000


In [5]:
# ==============================================================================
# CELL 4: TOP FUNCTIONS BY OWN TIME
# ==============================================================================
# Own time (tottime) = time spent inside the function itself, not in what it calls.
# Shows where the time really goes if a block above is not what dominates.
stats.sort_stats("tottime").print_stats(15);

Sun Oct 25 17:42:23 2026    /home/vroberto/FakeSpectra_DLAs/benchmarks/results/TNG50-2_profile/halo852.prof

         1847068 function calls (1844973 primitive calls) in 290.453 seconds

   Ordered by: internal time
   List reduced from 458 to 15 due to restriction <15>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
      131  108.046    0.825  108.052    0.825 /home/vroberto/miniconda3/envs/skewerjax_fakespectra/lib/python3.11/site-packages/h5py/_hl/dataset.py:824(__getitem__)
     1135  101.775    0.090  101.808    0.090 /home/vroberto/miniconda3/envs/skewerjax_fakespectra/lib/python3.11/site-packages/h5py/_hl/dataset.py:1083(read_direct)
      491   32.314    0.066   32.314    0.066 {built-in method fake_spectra._spectra_priv._near_lines}
     3687   17.664    0.005  119.473    0.032 {built-in method numpy.array}
      866   12.072    0.014   12.072    0.014 {built-in method fake_spectra._spectra_priv._Particle_Interpolate}
     1800    6.094    0.003    6.